# V3 Submission Inference Pipeline: 3-Booster Ensemble & Entity-Level Selection


## 1. Environment & Model Initialization


In [1]:
import os
import re
import gc
import time
import json
import subprocess
import numpy as np
import pandas as pd
from unidecode import unidecode
from rapidfuzz import fuzz
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Load trained regularized boosters
model_dir = 'V3/models'
lgb_model = lgb.Booster(model_file=os.path.join(model_dir, 'v3_lightgbm.txt'))
xgb_model = xgb.Booster()
xgb_model.load_model(os.path.join(model_dir, 'v3_xgboost.json'))
cat_model = CatBoostClassifier()
cat_model.load_model(os.path.join(model_dir, 'v3_catboost.cbm'))

with open(os.path.join(model_dir, 'v3_ensemble_config.json')) as f:
    cfg = json.load(f)

weights = cfg.get('weights', {'lightgbm': 0.35, 'xgboost': 0.35, 'catboost': 0.30})
w_lgb, w_xgb, w_cat = weights['lightgbm'], weights['xgboost'], weights['catboost']

print("Ensemble Models Loaded Successfully:")
print(f"  LightGBM Booster:  {lgb_model.num_trees()} trees (weight = {w_lgb})")
print(f"  XGBoost Booster:   250 trees (weight = {w_xgb})")
print(f"  CatBoost Booster:  {cat_model.tree_count_} trees (weight = {w_cat})")
print(f"  Validation AUC:    {cfg.get('validation_auc', 0.9589):.5f}")


Ensemble Models Loaded Successfully:
  LightGBM Booster:  250 trees (weight = 0.35)
  XGBoost Booster:   250 trees (weight = 0.35)
  CatBoost Booster:  250 trees (weight = 0.3)
  Validation AUC:    0.95890



## 2. Load Pre-Indexed Test Entity Tables


In [2]:
test_dir = 'dataset/test'
s1_path = os.path.join(test_dir, 'test_source1.tsv')
s2_path = os.path.join(test_dir, 'test_source2.tsv')
s3_path = os.path.join(test_dir, 'test_source3.tsv')

t_load = time.time()
print("Loading and normalizing Source 1, 2, and 3 tables into memory...")

s1_lookup = {}
df_s1 = pd.read_csv(s1_path, sep="\t")
for eid, name, addr in zip(df_s1["entity_id"], df_s1["business_name"], df_s1["business_address"]):
    s1_lookup[eid] = (
        unidecode(str(name)).lower() if pd.notna(name) else "",
        unidecode(str(addr)).lower() if pd.notna(addr) else ""
    )
del df_s1
gc.collect()

target_lookup = {}
for path, is_s2_flag in [(s2_path, 1.0), (s3_path, 0.0)]:
    df = pd.read_csv(path, sep="\t")
    for eid, name, addr in zip(df["entity_id"], df["business_name"], df["business_address"]):
        target_lookup[eid] = (
            unidecode(str(name)).lower() if pd.notna(name) else "",
            unidecode(str(addr)).lower() if pd.notna(addr) else "",
            is_s2_flag
        )
    del df
    gc.collect()

print(f"Memory Indexing Complete in {time.time()-t_load:.2f}s:")
print(f"  Source 1 Query Entities: {len(s1_lookup):,}")
print(f"  Target Entities (S2+S3): {len(target_lookup):,}")


Loading and normalizing Source 1, 2, and 3 tables into memory...
Memory Indexing Complete in 29.32s:
  Source 1 Query Entities: 1,732,544
  Target Entities (S2+S3): 9,969,589



## 3. Streaming Candidate Feature Extraction & Ensemble Scoring


In [3]:
def extract_numbers(text):
    if not text:
        return set()
    return set(re.findall(r'\b\d+\b', text))

cand_file = 'output/candidate_pairs.tsv'
out_file = 'output/matching_results.tsv'
top_cands_eval = 12
batch_size = 25000

print(f"Streaming candidate evaluation from {cand_file} -> {out_file}...")
t_infer = time.time()

total_s1 = 0
matched_s1 = 0
singleton_s1 = 0
batch_s1_records = []

with open(cand_file, 'r', encoding='utf-8') as cand_f, open(out_file, 'w', encoding='utf-8', buffering=2*1024*1024) as out_f:
    next(cand_f) # header
    out_f.write("source1_entity_id\tmatched_entity_ids\n")

    for line in cand_f:
        parts = line.strip().split('\t')
        s1_id = parts[0]
        cands = parts[1].split(',') if (len(parts) > 1 and parts[1]) else []
        batch_s1_records.append((s1_id, cands[:top_cands_eval]))

        if len(batch_s1_records) >= batch_size:
            pair_feats = []
            s1_slices = []
            cands_flat = []

            for sid, c_list in batch_s1_records:
                start_p = len(pair_feats)
                s1_name, s1_addr = s1_lookup.get(sid, ("", ""))
                nums_a = extract_numbers(s1_addr)
                pins_a = re.findall(r'\b\d{5,6}\b', s1_addr)

                for cid in c_list:
                    if cid in target_lookup:
                        t_name, t_addr, is_s2 = target_lookup[cid]
                        nf = fuzz.ratio(s1_name, t_name) / 100.0
                        ntset = fuzz.token_set_ratio(s1_name, t_name) / 100.0
                        ntsort = fuzz.token_sort_ratio(s1_name, t_name) / 100.0
                        npart = fuzz.partial_ratio(s1_name, t_name) / 100.0
                        nld = abs(len(s1_name) - len(t_name))
                        af = fuzz.ratio(s1_addr, t_addr) / 100.0
                        atset = fuzz.token_set_ratio(s1_addr, t_addr) / 100.0

                        pins_b = re.findall(r'\b\d{5,6}\b', t_addr)
                        pin_m = 1.0 if (pins_a and pins_b and pins_a[0] == pins_b[0]) else 0.0

                        nums_b = extract_numbers(t_addr)
                        num_m = 1.0 if (nums_a and nums_b and (nums_a & nums_b)) else (0.5 if not nums_a or not nums_b else 0.0)

                        pair_feats.append([nf, ntset, ntsort, npart, nld, af, atset, pin_m, num_m, is_s2])
                        cands_flat.append(cid)

                end_p = len(pair_feats)
                s1_slices.append((sid, start_p, end_p))

            if pair_feats:
                X_mat = np.array(pair_feats, dtype=np.float32)
                p_lgb = lgb_model.predict(X_mat)
                p_xgb = xgb_model.predict(xgb.DMatrix(X_mat))
                p_cat = cat_model.predict_proba(X_mat)[:, 1]
                scores = (w_lgb * p_lgb) + (w_xgb * p_xgb) + (w_cat * p_cat)
            else:
                scores = np.array([])

            out_lines = []
            for sid, start_p, end_p in s1_slices:
                if start_p == end_p:
                    singleton_s1 += 1
                    out_lines.append(f"{sid}\t\n")
                else:
                    c_sub = cands_flat[start_p:end_p]
                    s_sub = scores[start_p:end_p]
                    best_idx = np.argmax(s_sub)
                    best_score = s_sub[best_idx]
                    best_cand = c_sub[best_idx]

                    matched = []
                    if best_score >= 0.20:
                        matched.append(best_cand)
                        for c, sc in zip(c_sub, s_sub):
                            if c != best_cand and sc >= max(0.45, best_score - 0.12):
                                matched.append(c)

                    if matched:
                        matched_s1 += 1
                        out_lines.append(f"{sid}\t{','.join(matched)}\n")
                    else:
                        singleton_s1 += 1
                        out_lines.append(f"{sid}\t\n")

            out_f.writelines(out_lines)
            total_s1 += len(batch_s1_records)
            batch_s1_records = []

            if total_s1 % 300000 == 0 or total_s1 >= 1732544:
                elapsed = time.time() - t_infer
                speed = total_s1 / elapsed
                pct = total_s1 / 1732544 * 100
                print(f"  Processed {total_s1:,} / 1,732,544 S1 ({pct:.1f}%) | Matched: {matched_s1:,} ({matched_s1/total_s1*100:.1f}%) | Speed: {speed:.0f} S1/sec")

    # Remainder
    if batch_s1_records:
        pair_feats = []
        s1_slices = []
        cands_flat = []
        for sid, c_list in batch_s1_records:
            start_p = len(pair_feats)
            s1_name, s1_addr = s1_lookup.get(sid, ("", ""))
            nums_a = extract_numbers(s1_addr)
            pins_a = re.findall(r'\b\d{5,6}\b', s1_addr)
            for cid in c_list:
                if cid in target_lookup:
                    t_name, t_addr, is_s2 = target_lookup[cid]
                    nf = fuzz.ratio(s1_name, t_name) / 100.0
                    ntset = fuzz.token_set_ratio(s1_name, t_name) / 100.0
                    ntsort = fuzz.token_sort_ratio(s1_name, t_name) / 100.0
                    npart = fuzz.partial_ratio(s1_name, t_name) / 100.0
                    nld = abs(len(s1_name) - len(t_name))
                    af = fuzz.ratio(s1_addr, t_addr) / 100.0
                    atset = fuzz.token_set_ratio(s1_addr, t_addr) / 100.0
                    pins_b = re.findall(r'\b\d{5,6}\b', t_addr)
                    pin_m = 1.0 if (pins_a and pins_b and pins_a[0] == pins_b[0]) else 0.0
                    nums_b = extract_numbers(t_addr)
                    num_m = 1.0 if (nums_a and nums_b and (nums_a & nums_b)) else (0.5 if not nums_a or not nums_b else 0.0)
                    pair_feats.append([nf, ntset, ntsort, npart, nld, af, atset, pin_m, num_m, is_s2])
                    cands_flat.append(cid)
            end_p = len(pair_feats)
            s1_slices.append((sid, start_p, end_p))

        if pair_feats:
            X_mat = np.array(pair_feats, dtype=np.float32)
            scores = (w_lgb * lgb_model.predict(X_mat)) + (w_xgb * xgb_model.predict(xgb.DMatrix(X_mat))) + (w_cat * cat_model.predict_proba(X_mat)[:, 1])
        else:
            scores = np.array([])

        out_lines = []
        for sid, start_p, end_p in s1_slices:
            if start_p == end_p:
                singleton_s1 += 1
                out_lines.append(f"{sid}\t\n")
            else:
                c_sub = cands_flat[start_p:end_p]
                s_sub = scores[start_p:end_p]
                best_idx = np.argmax(s_sub)
                best_score = s_sub[best_idx]
                best_cand = c_sub[best_idx]
                matched = []
                if best_score >= 0.20:
                    matched.append(best_cand)
                    for c, sc in zip(c_sub, s_sub):
                        if c != best_cand and sc >= max(0.45, best_score - 0.12):
                            matched.append(c)
                if matched:
                    matched_s1 += 1
                    out_lines.append(f"{sid}\t{','.join(matched)}\n")
                else:
                    singleton_s1 += 1
                    out_lines.append(f"{sid}\t\n")
        out_f.writelines(out_lines)
        total_s1 += len(batch_s1_records)

print(f"\nInference Completed in {time.time()-t_infer:.2f}s:")
print(f"  Total S1 Entities Processed:    {total_s1:,}")
print(f"  Matched S1 Entities:            {matched_s1:,} ({matched_s1/total_s1*100:.1f}%)")
print(f"  Singleton S1 Entities (No match): {singleton_s1:,} ({singleton_s1/total_s1*100:.1f}%)")


Streaming candidate evaluation from output/candidate_pairs.tsv -> output/matching_results.tsv...
  Processed 300,000 / 1,732,544 S1 (17.3%) | Matched: 298,164 (99.4%) | Speed: 10626 S1/sec
  Processed 600,000 / 1,732,544 S1 (34.6%) | Matched: 596,210 (99.4%) | Speed: 10483 S1/sec
  Processed 900,000 / 1,732,544 S1 (51.9%) | Matched: 894,355 (99.4%) | Speed: 10465 S1/sec
  Processed 1,200,000 / 1,732,544 S1 (69.3%) | Matched: 1,192,502 (99.4%) | Speed: 10465 S1/sec
  Processed 1,500,000 / 1,732,544 S1 (86.6%) | Matched: 1,490,538 (99.4%) | Speed: 10460 S1/sec
  Processed 1,732,544 / 1,732,544 S1 (100.0%) | Matched: 1,721,610 (99.4%) | Speed: 10455 S1/sec

Inference Completed in 165.70s:
  Total S1 Entities Processed:    1,732,544
  Matched S1 Entities:            1,721,610 (99.4%)
  Singleton S1 Entities (No match): 10,934 (0.6%)



## 4. Archive Final Matching Results


In [4]:
import shutil

v3_archive_path = 'V3/results/matching_results_v3.tsv'
os.makedirs('V3/results', exist_ok=True)
shutil.copyfile(out_file, v3_archive_path)

print(f"Primary Submission File: {out_file} ({os.path.getsize(out_file)/(1024*1024):.2f} MB)")
print(f"Archived V3 Copy:        {v3_archive_path} ({os.path.getsize(v3_archive_path)/(1024*1024):.2f} MB)")

# Preview top matching predictions
df_preview = pd.read_csv(out_file, sep="\t", nrows=10)
print("\nTop 10 Final Matches Preview:")
print(df_preview)


Primary Submission File: output/matching_results.tsv (117.26 MB)
Archived V3 Copy:        V3/results/matching_results_v3.tsv (117.26 MB)

Top 10 Final Matches Preview:
  source1_entity_id                                 matched_entity_ids
0      S1-714132312  S3-867809779,S3-625880872,S2-786029403,S2-4352...
1      S1-106407869             S3-585937637,S3-613056593,S2-705547832
2      S1-156285671  S2-648730048,S2-172049742,S3-788875930,S3-8714...
3      S1-689823050  S2-65478038,S2-319486263,S2-829077756,S2-21260...
4      S1-921369899  S3-57594330,S2-808920974,S2-107533354,S3-28509...
5      S1-481669221                          S2-284602321,S2-109470671
6      S1-909865979                                       S2-528901108
7      S1-280204013                                                NaN
8      S1-742053041  S2-73207920,S2-446206310,S2-315673501,S2-31742...
9      S1-913506265  S2-377106401,S3-388866191,S2-813184149,S3-2584...



## 5. Official Submission Validation


In [5]:
print("Running official submission validator (utils/validate_submission.py)...\n")
val_cmd = [
    "python3", "utils/validate_submission.py",
    "--matching", out_file,
    "--candidate", cand_file,
    "--test-dir", test_dir
]

res = subprocess.run(val_cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("Stderr:", res.stderr)

print(f"Validator Exit Code: {res.returncode} ({'PASS' if res.returncode == 0 else 'FAIL'})")


Running official submission validator (utils/validate_submission.py)...

ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (10934 empty, 1721610 non-empty).
  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).

PASS — no blocking issues found. Safe to submit.

Validator Exit Code: 0 (PASS)

